# INICIANDO


In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import re

catalog = "sbsrisk_dev"

t24 = f"{catalog}.bronze.bronze_sbs_morosidad_2024"
t25 = f"{catalog}.bronze.bronze_sbs_morosidad_2025"

df24 = spark.table(t24).withColumn("anio", F.lit(2024))
df25 = spark.table(t25).withColumn("anio", F.lit(2025))

dfb = df24.unionByName(df25, allowMissingColumns=True)

# Normaliza strings tipo "nan" -> null
data_cols = [c for c in dfb.columns if c not in ["source_file","periodo_mes","sheet_title","sheet_num","anio"]]
for c in data_cols:
    dfb = dfb.withColumn(
        c,
        F.when(F.lower(F.col(c).cast("string")).isin("nan","none",""), None).otherwise(F.col(c))
    )

In [0]:
df13 = dfb.filter(F.col("sheet_num")==13)
df16 = dfb.filter(F.col("sheet_num")==16)

print("rows sheet13:", df13.count())
print("rows sheet16:", df16.count())

In [0]:
def to_double(col):
    # limpia %, comas, espacios
    s = F.regexp_replace(F.col(col).cast("string"), "%", "")
    s = F.regexp_replace(s, r"\s+", "")
    # cambia coma decimal a punto si viniera así
    s = F.regexp_replace(s, ",", ".")
    return s.cast("double")

# Detecta columnas tipo "0","1","2"... (tu DF viene así desde pandas)
grid_cols_13 = [c for c in df13.columns if re.fullmatch(r"\d+", c)]
grid_cols_13 = sorted(grid_cols_13, key=lambda x: int(x))

# c0 = nombre CMAC normalmente
c0 = grid_cols_13[0] if len(grid_cols_13)>0 else None
if c0 is None:
    raise Exception("No encontré columnas numéricas tipo '0','1',... en sheet 13")

df13_cmac = (
    df13
    .withColumn("cmac", F.col(c0).cast("string"))
    .filter(F.col("cmac").rlike(r"^CMAC\s"))
)

# intenta castear el resto de columnas a double (si no se puede, quedará null)
value_cols = []
for c in grid_cols_13[1:]:
    df13_cmac = df13_cmac.withColumn(f"v_{c}", to_double(c))
    value_cols.append(f"v_{c}")

df13_silver = df13_cmac.select(
    "anio","periodo_mes","cmac","source_file","sheet_title","sheet_num",
    *value_cols
)

display(df13_silver.limit(30))

In [0]:
from pyspark.sql import functions as F

grid_cols_16 = [c for c in df16.columns if re.fullmatch(r"\d+", c)]
grid_cols_16 = sorted(grid_cols_16, key=lambda x: int(x))

# Creamos una columna "row_text" para detectar la fila header
df16_txt = df16.withColumn(
    "row_text",
    F.lower(F.concat_ws(" | ", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in grid_cols_16[:20]]))
)

# Nos quedamos solo con filas que parezcan parte de la tabla (Concepto o valores)
df16_tab = df16_txt.filter(
    F.col("row_text").contains("concepto") |
    F.col(grid_cols_16[0]).cast("string").rlike(r"(?i)tarjetas|prestamos|factoring|otros|corporativos")
)

display(df16_tab.select(*grid_cols_16[:10], "row_text").limit(30))

In [0]:
# ========== SHEET 16 SILVER CORRECTO ==========

# columnas tipo "0","1","2"... 
grid_cols_16 = [c for c in df16.columns if re.fullmatch(r"\d+", c)]
grid_cols_16 = sorted(grid_cols_16, key=lambda x: int(x))

concepto_col = "1"     # confirmado por tu captura
cmac_cols = grid_cols_16[2:]   # desde columna 2 en adelante

# 1️⃣ Detectar fila header real (donde Concepto está y hay nombres CMAC)
header_row = (
    df16
    .filter(F.col(concepto_col) == "Concepto")
    .limit(1)
    .collect()
)

if not header_row:
    raise Exception("No encontré fila header con 'Concepto'")

header_row = header_row[0]

# Mapeo: columna -> nombre CMAC real
cmac_mapping = {}
for c in cmac_cols:
    val = header_row[c]
    if val and str(val).startswith("CMAC"):
        cmac_mapping[c] = val.strip()

# 2️⃣ Filas reales de datos (conceptos)
df16_data = (
    df16
    .filter(F.col(concepto_col).isNotNull())
    .filter(~F.col(concepto_col).isin("Concepto","(En porcentaje)"))
)

# 3️⃣ Unpivot dinámico usando mapping real
stack_expr = "stack({0}, {1}) as (cmac, valor)".format(
    len(cmac_mapping),
    ", ".join([f"'{cmac_mapping[c]}', `{c}`" for c in cmac_mapping.keys()])
)

df16_long = (
    df16_data
    .selectExpr(
        "anio",
        "periodo_mes",
        "source_file",
        "sheet_title",
        "sheet_num",
        f"`{concepto_col}` as concepto",
        stack_expr
    )
    .withColumn(
        "valor",
        F.regexp_replace(F.col("valor").cast("string"), "%", "")
    )
    .withColumn(
        "valor",
        F.regexp_replace(F.col("valor"), ",", ".")
    )
    .withColumn("valor", F.col("valor").cast("double"))
)

display(df16_long.limit(30))
print("Filas sheet16 long:", df16_long.count())

# ========== LIMPIEZA FINAL SHEET 16 (DEJAR SOLO CONCEPTOS REALES) ==========

df16_clean = (
    df16_long
    # normaliza texto
    .withColumn("concepto", F.trim(F.col("concepto")))
    .withColumn("cmac", F.trim(F.col("cmac")))
    # quita basura típica
    .filter(~F.lower(F.col("concepto")).rlike(r"^(volver al índice|nan|null)$"))
    .filter(~F.lower(F.col("concepto")).rlike(r"^(nota\s*\d+|nota\s*1|nota\s*2)"))
    .filter(~F.lower(F.col("concepto")).rlike(r"resoluci"))
    .filter(~F.lower(F.col("concepto")).rlike(r"reglamento|cap[ií]tulo|numeral"))
    # quita filas que son fechas tipo 2024-04-30 00:00:00
    .filter(~F.col("concepto").rlike(r"^\d{4}-\d{2}-\d{2}"))
    # solo valores válidos
    .filter(F.col("valor").isNotNull())
)

# (opcional) quitar "**" al final de CMAC o concepto
df16_clean = (
    df16_clean
    .withColumn("cmac", F.regexp_replace(F.col("cmac"), r"\s*\*\*$", ""))
    .withColumn("concepto", F.regexp_replace(F.col("concepto"), r"\s*\*\*$", ""))
)

display(df16_clean.limit(30))
print("Filas sheet16 clean:", df16_clean.count())

In [0]:
from pyspark.sql import functions as F

# 1) Normaliza concepto
df16_tmp = (
    df16_long
    .withColumn("concepto", F.trim(F.regexp_replace(F.col("concepto"), r"^\*+\s*", "")))  # quita ** al inicio
    .withColumn("concepto_l", F.lower(F.col("concepto")))
)

# 2) Filtros para BOTAR filas "basura"
df16_clean = (
    df16_tmp
    # quita fechas tipo 2024-01-31 00:00:00
    .filter(~F.col("concepto").rlike(r"^\d{4}-\d{2}-\d{2}"))
    # quita textos de notas/resoluciones/volver al índice/etc
    .filter(~F.col("concepto_l").rlike(r"resoluci|reglamento|nota\s*\d|incluye|volver al"))
    # quita filas vacías o encabezados repetidos
    .filter(F.col("concepto").isNotNull())
    .filter(~F.col("concepto").isin("Concepto", "(En porcentaje)"))
    # me quedo solo con filas que tienen dato numérico
    .filter(F.col("valor").isNotNull())
    # (opcional pero recomendable) asegura CMAC válida
    .filter(F.col("cmac").rlike(r"^CMAC\s"))
    .drop("concepto_l")
)

display(df16_clean.select("concepto").distinct().orderBy("concepto").limit(60))
df16_clean.count()

# Lista "golden" de conceptos válidos (ajusta si aparece alguno nuevo real)
CONCEPTOS_OK = [
  "Créditos hipotecarios para vivienda",
  "Créditos a grandes empresas",
  "Tarjetas de crédito",
  "Créditos corporativos",
  "Créditos corporativos **",
  "Créditos a microempresas",
  "Créditos a microempresas **",
  "Créditos a medianas empresas",
  "Créditos pequeñas empresas",
  "Créditos pequeñas empresas **",
  "Préstamos no revolventes",
  "Créditos de consumo",
  "Total Créditos Directos",
  "Descuentos",
  "Factoring",
  "Arrendamiento financiero y Lease-back *",
  "Comercio exterior",
  "Otros 1/",
  "Pignoraticios",
  "Préstamos",
  "Préstamos Mivivienda",
  "Préstamos autos"
]

# Normaliza los ** dentro del texto para comparar
df16_clean = df16_clean.withColumn("concepto", F.regexp_replace("concepto", r"\s+\*\*+$", " **"))

df16_clean = df16_clean.filter(F.col("concepto").isin(CONCEPTOS_OK))

In [0]:
from pyspark.sql import functions as F

# Limpieza de conceptos "basura" en sheet 16 (notas, fechas, resoluciones, etc.)
df16_clean = (
    df16_long
    # 1) nos quedamos con filas que realmente tienen valor numérico
    .filter(F.col("valor").isNotNull())
    # 2) concepto debe existir y no ser vacío
    .filter(F.col("concepto").isNotNull())
    .withColumn("concepto", F.trim(F.col("concepto")))
    .filter(F.length(F.col("concepto")) > 0)
    # 3) filtra conceptos que claramente NO son categorías (notas/encabezados)
    .filter(~F.lower(F.col("concepto")).rlike(
        r"^(volver al índice|nota|información|resolución|capítulo|numeral|cuadro|balance|comprobación)"
    ))
    # 4) filtra líneas que suelen ser solo texto contextual
    .filter(~F.lower(F.col("concepto")).rlike(
        r"(la información contenida|se complementa|definiciones|reglamento|obt(enida|enido))"
    ))
    # 5) evita que “concepto” sea una fecha (por si entra como texto)
    .filter(~F.col("concepto").rlike(r"^\d{4}-\d{2}-\d{2}"))
)

print("Filas df16_long:", df16_long.count())
print("Filas df16_clean:", df16_clean.count())
display(df16_clean.select("concepto").distinct().orderBy("concepto").limit(80))

In [0]:
spark.sql(f"USE CATALOG {catalog}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

(df13_silver.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable(f"{catalog}.silver.silver_morosidad_dias_wide"))

(df16_clean.write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable(f"{catalog}.silver.silver_morosidad_tipo_long"))

# Validaciones

In [0]:
df16_clean.count()

In [0]:
# 1) ¿Cuántas CMAC detectó?
display(df16_clean.groupBy("cmac").count().orderBy("cmac"))

# 2) ¿Cuántos conceptos distintos?
display(df16_clean.select("concepto").distinct().orderBy("concepto"))

# 3) ¿Hay valores nulos o filas raras?
display(
  df16_clean
    .select(
      F.count("*").alias("total"),
      F.sum(F.col("valor").isNull().cast("int")).alias("valor_nulls"),
      F.sum(F.col("cmac").isNull().cast("int")).alias("cmac_nulls"),
      F.sum(F.col("concepto").isNull().cast("int")).alias("concepto_nulls")
    )
)

In [0]:
%sql
SELECT COUNT(*) FROM sbsrisk_dev.silver.silver_morosidad_dias_wide;
SELECT COUNT(*) FROM sbsrisk_dev.silver.silver_morosidad_tipo_long;

In [0]:
display(df16_clean.select("concepto").distinct().limit(50))